# Circuit Diagram Explanation Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B explaining circuits in plain language, generating SPICE
netlists, and answering what-if questions. Offline on T4 / CPU via GGUF.

In [ ]:
# GPU compatibility check — P100 (sm_60) not supported by PyTorch 2.10+cu128
import torch
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    if major < 7:
        name = torch.cuda.get_device_name()
        raise RuntimeError(
            f"\n{'='*60}\n"
            f"INCOMPATIBLE GPU: {name} (sm_{major}{minor})\n"
            f"PyTorch 2.10+cu128 requires sm_70+ (T4, V100, A100).\n"
            f"\nFix: Kaggle > Settings > Accelerator > select 'T4 GPU x1' > Save > Restart\n"
            f"{'='*60}"
        )
print(f"GPU OK: {torch.cuda.get_device_name()} sm_{torch.cuda.get_device_capability()[0]}{torch.cuda.get_device_capability()[1]}")

In [ ]:
# Cell 1 — install and clone (training data is inside the repo)
import subprocess, sys, os
subprocess.run(["pip", "install", "unsloth[kaggle-new]", "trl", "datasets", "-q"], check=False)
subprocess.run(["git", "clone", "https://github.com/govindrathore27/gemma4-engineering-diagrams.git",
                "/kaggle/working/repo"], capture_output=True)
subprocess.run(["git", "-C", "/kaggle/working/repo", "pull"], capture_output=True)
sys.path.insert(0, "/kaggle/working/repo")

TRAIN_JSONL = "/kaggle/working/repo/project2_circuit_tutor/data/train.jsonl"
EVAL_JSONL  = "/kaggle/working/repo/project2_circuit_tutor/data/eval.jsonl"

n_train = sum(1 for _ in open(TRAIN_JSONL))
n_eval  = sum(1 for _ in open(EVAL_JSONL))
print(f"Training data ready: {n_train} train rows, {n_eval} eval rows")

In [ ]:
# Cell 2 — fine-tune Gemma 4 E4B with LoRA
os.environ["WANDB_DISABLED"] = "true"
from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="circuit",
    data_path=TRAIN_JSONL,
    output_dir="/kaggle/working/circuit_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 3 — evaluate
import json
from pathlib import Path
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/circuit_adapter")
eval_rows = [json.loads(l) for l in open(EVAL_JSONL, encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected    = [r["output"] for r in eval_rows]
bleu = evaluate_batch(predictions, expected, metric="bleu")
f1   = evaluate_batch(predictions, expected, metric="f1")
print(f"BLEU-1: {bleu['mean']:.3f}  |  Token F1: {f1['mean']:.3f}")
Path("/kaggle/working/eval_results.txt").write_text(
    f"BLEU-1: {bleu['mean']:.3f}\nToken F1: {f1['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 4 — GGUF export
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/circuit_adapter",
    output_path="/kaggle/working/circuit_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 5 — demo
ctx = "Components: 2x resistor, 1x capacitor, 1x diode"
print("=== Circuit Diagram Explanation Tutor Demo ===\n")
for q in [
    "Describe what this circuit does based on its components.",
    "What limits the current through the diode in this circuit?",
    "Generate a SPICE-compatible netlist for this circuit.",
    "If all resistors were doubled in value, how would that affect the circuit?",
]:
    print(f"Q: {q}\nA: {run(model, tokenizer, q, ctx)}\n")

## Impact

100M+ electronics students in underserved communities have no access to LTspice/Multisim.
This model explains any circuit in plain language and generates SPICE netlists from
component annotations — fully offline via llama.cpp on any laptop.

**Training data:** 3,484 QA pairs from circuit1k (1,000 annotated circuit images, YOLO format).